## Example 1: Basic raster I/O

In [ ]:
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_bounds
import numpy as np
import tempfile
import os

# Example 1: Basic raster I/O
print("Testing rasterio basic I/O:")

data = np.random.randint(0, 255, (100, 100), dtype=np.uint8)
crs = CRS.from_epsg(4326)
transform = from_bounds(-120, 35, -119, 36, 100, 100)

with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as f:
    path = f.name

with rasterio.open(path, 'w', driver='GTiff', height=100, width=100, count=1, dtype=np.uint8, crs=crs, transform=transform) as dst:
    dst.write(data, 1)

with rasterio.open(path) as src:
    print(f"✓ Shape: {src.shape}, CRS: {src.crs}")
    assert src.shape == (100, 100) and src.count == 1
    assert src.crs == crs and src.transform == transform
    assert (src.read(1) == data).all()

os.unlink(path)

## Example 2: Windowed reading

In [ ]:
import rasterio
from rasterio.windows import Window
import numpy as np
import tempfile

# Example 2: Windowed reading
print("Testing rasterio windowed reading:")

# Create multi-band data
data = np.random.randint(0, 255, (3, 200, 200), dtype=np.uint8)

with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as f:
    path = f.name

with rasterio.open(path, 'w', driver='GTiff', height=200, width=200, count=3, dtype=np.uint8) as dst:
    dst.write(data)

# Read windows
with rasterio.open(path) as src:
    # Read center window
    window = Window(75, 75, 50, 50)
    window_data = src.read(window=window)
    print(f"✓ Full dataset: {src.shape}, Window: {window_data.shape}")
    assert window_data.shape == (3, 50, 50)
    assert (window_data == data[:, 75:125, 75:125]).all()
    
import os; os.unlink(path)

## Example 3: CRS reprojection

In [ ]:
import rasterio
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, transform_bounds
import numpy as np
import tempfile
import os

# Example 3: CRS reprojection
print("Testing rasterio CRS reprojection:")

# A constant-valued raster in geographic coordinates (EPSG:4326); 0 is the nodata value
data = np.full((100, 100), 7, dtype=np.uint8)
src_crs = CRS.from_epsg(4326)
bounds = (-120, 35, -119, 36)
src_transform = from_bounds(*bounds, 100, 100)
dst_crs = CRS.from_epsg(3857)

dst_transform, width, height = calculate_default_transform(src_crs, dst_crs, 100, 100, *bounds)
reprojected = np.zeros((height, width), dtype=np.uint8)
reproject(
    source=data, destination=reprojected,
    src_transform=src_transform, src_crs=src_crs, src_nodata=0,
    dst_transform=dst_transform, dst_crs=dst_crs, dst_nodata=0,
    resampling=Resampling.nearest,
)
valid = reprojected[reprojected != 0]
print(f"✓ {src_crs} (100, 100) -> {dst_crs} {reprojected.shape}, {valid.size} valid pixels")
assert reprojected.shape == (height, width) and valid.size > 0
assert (valid == 7).all()

# Write the result and read the CRS back
with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as f:
    path = f.name
with rasterio.open(path, 'w', driver='GTiff', height=height, width=width, count=1, dtype=np.uint8,
                   crs=dst_crs, transform=dst_transform, nodata=0) as dst:
    dst.write(reprojected, 1)
with rasterio.open(path) as src:
    assert src.crs == dst_crs and src.shape == (height, width)
    left, bottom, right, top = src.bounds
    print(f"✓ Web Mercator bounds: ({left:.0f}, {bottom:.0f}, {right:.0f}, {top:.0f})")
    # The output covers the source extent transformed to EPSG:3857, to within one output pixel
    expected = transform_bounds(src_crs, dst_crs, *bounds)
    assert np.allclose(src.bounds, expected, atol=abs(dst_transform.a)), (src.bounds, expected)

os.unlink(path)

## Example 4: Raster masking

In [ ]:
import rasterio
from rasterio.mask import mask
from rasterio.crs import CRS
from rasterio.transform import from_bounds
import numpy as np
import tempfile
import os

# Example 4: Raster masking
print("Testing rasterio masking:")

data = np.random.randint(1, 6, (100, 100), dtype=np.uint8)
crs = CRS.from_epsg(3857)
transform = from_bounds(-8240000, 4970000, -8220000, 4990000, 100, 100)

with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as f:
    path = f.name

with rasterio.open(path, 'w', driver='GTiff', height=100, width=100, count=1, dtype=np.uint8, crs=crs, transform=transform) as dst:
    dst.write(data, 1)

circle = {"type": "Polygon", "coordinates": [[(-8230000+i*5000, 4980000+j*5000) for i, j in [(1,0), (0,1), (-1,0), (0,-1), (1,0)]]]}

with rasterio.open(path) as src:
    masked, _ = mask(src, [circle], crop=True, nodata=0)
    print(f"✓ Masked: {src.shape} -> {masked.shape[1:]}")
    assert masked.shape[0] == 1 and masked.shape[1] < 100 and masked.shape[2] < 100
    assert (masked[masked != 0] >= 1).all() and (masked == 0).any()   # outside the polygon is nodata

os.unlink(path)

## Example 5: Virtual raster and resampling

In [ ]:
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
import numpy as np
import tempfile

# Example 5: Virtual raster and resampling
print("Testing rasterio VRT:")

# Create source data
data = np.random.randint(0, 255, (100, 100), dtype=np.uint8)

with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as f:
    path = f.name

with rasterio.open(path, 'w', driver='GTiff', height=100, width=100, count=1, dtype=np.uint8) as dst:
    dst.write(data, 1)

# Use VRT to resample
with rasterio.open(path) as src:
    with WarpedVRT(src, crs=src.crs, resampling=Resampling.bilinear, width=50, height=50) as vrt:
        resampled = vrt.read(1)
        print(f"✓ Original: {src.shape}, Resampled: {resampled.shape}")
        assert resampled.shape == (50, 50)

import os; os.unlink(path)